In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np

from sklearn.metrics import accuracy_score, f1_score

from src.datasets import get_dataloaders
from src.models import ResNet18Classifier

In [2]:
DATA_DIR = "/home/divye/Desktop/cnn/data/2750"

data = get_dataloaders(DATA_DIR, batch_size=32)

if isinstance(data, dict):
    train_loader = data["train_loader"]
    val_loader = data["val_loader"]
    classes = data["classes"]
else:
    train_loader, val_loader, classes, class_to_idx = data

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ResNet18Classifier(10)

model.load_state_dict(
    torch.load(
        "../checkpoints/resnet18_best.pt",
        map_location=device
    )
)

model.to(device)
model.eval()

ResNet18Classifier(
  (model): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1

In [3]:
preds = []
labels = []

with torch.no_grad():

    for images, target in val_loader:

        images = images.to(device)

        outputs = model(images)

        pred = outputs.argmax(1)

        preds.extend(pred.cpu().numpy())
        labels.extend(target.numpy())

random_acc = accuracy_score(labels, preds)
random_f1 = f1_score(labels, preds, average="macro")

print("=" * 50)
print("Random Split Results")
print("=" * 50)
print(f"Accuracy : {random_acc:.4f}")
print(f"Macro F1 : {random_f1:.4f}")


Random Split Results
Accuracy : 0.9698
Macro F1 : 0.9692


In [4]:
print("=" * 60)
print("Spatial Leakage Experiment")
print("=" * 60)

print(f"Random Split Accuracy      : {random_acc:.4f}")
print(f"Random Split Macro F1      : {random_f1:.4f}")

print()
print("Grouped Spatial Split      : Not Available")
print()

print("Reason:")
print("- EuroSAT RGB does not contain GPS coordinates.")
print("- No tile adjacency is provided.")
print("- No region identifiers are available.")
print("- Therefore, a true geographic block split cannot be reproduced.")

Spatial Leakage Experiment
Random Split Accuracy      : 0.9698
Random Split Macro F1      : 0.9692

Grouped Spatial Split      : Not Available

Reason:
- EuroSAT RGB does not contain GPS coordinates.
- No tile adjacency is provided.
- No region identifiers are available.
- Therefore, a true geographic block split cannot be reproduced.
